# Lab 10-03 — Evidence verification loop (SciFact claims)

**Track 10 · Agentic RAG** — how we check whether retrieved evidence actually backs the answer.

Generation metrics grade *what* the model said; attribution metrics grade *whether* the retrieved evidence supports it. This lab implements the evidence-verification half of that idea with `tools/verifier.py` over the SciFact corpus: for each claim, embed + rank the 5183-doc corpus, retrieve top-5 candidate evidence passages, then ask the local LLM (via `json_object`) for a three-way verdict.

```text
4 claims + 5183-doc SciFact corpus
  -> BGE embed once (CPU, ~5-6 min)
  -> cosine top-5 evidence per claim
  -> OllamaLLM json_object verdict (qwen2.5-coder:7b, local)
  -> SUPPORTED / REFUTED / NOT_ENOUGH_INFO rows
```

The corpus is embedded exactly ONCE and cached; per-claim retrieval is then a pure cosine rank over cached vectors. The loop lives in `tools/verifier.py` (LangChain has no native evidence-verification agent, so this repo ships its own). Notice what is *missing* from the pipeline: no gold-label matching, no benchmark scoring. A local 7B on 4 claims is not a benchmark run; it is a pipeline-integrity check.


## Setup

This notebook mirrors `curriculum/10-agentic-rag/03-scifact-verify.py` exactly — the same verified code, split into cells. Two prerequisites must hold before it will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM every verdict call goes to (`llms/ollama.py` talks to it through `langchain-ollama`). Fully local: no API key, no quota. If the server is not up, every verdict call fails.
- **Both data files on disk** — `Data/corpus/scifact/data/corpus.jsonl` (5183 docs) and `Data/corpus/scifact/data/claims_dev.jsonl`, already fetched by the repo's manifest-verified fetchers.

The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. From the terminal the lab runs as:

```bash
python curriculum/10-agentic-rag/03-scifact-verify.py          # run + demo
python curriculum/10-agentic-rag/03-scifact-verify.py --verify # verification gate
```

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).

⚠️ **The corpus embed takes ~5-6 minutes on CPU** — that is the single slow step. Everything after it (the per-claim LLM verdicts) is fast.


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   langchain-ollama   -> the Ollama chat backend behind llms/ollama.py
#   langchain-huggingface -> sentence-transformer embeddings via embeddings/bge.py
%pip install langchain-ollama langchain-huggingface


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import json  # noqa: E402
import time  # noqa: E402
from collections import Counter  # noqa: E402

from embeddings.bge import BGEEmbedding  # noqa: E402
from llms.ollama import OllamaLLM  # noqa: E402
from tools.verifier import VALID_VERDICTS, verify_loop  # noqa: E402


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_CLAIMS = 4` takes the **deterministic head** of the SciFact dev claims — and because every claim costs exactly one LLM verdict call, this number *is* the runtime knob: 4 claims, ~5-8 local LLM calls, no sampling variance between runs. `TOP_K = 5` controls how many candidate evidence passages each claim retrieves. `BGE_DEVICE = "cpu"` keeps embeddings off the GPU so Ollama keeps all the VRAM.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
CORPUS_PATH = Path("Data/corpus/scifact/data/corpus.jsonl")
CLAIMS_PATH = Path("Data/corpus/scifact/data/claims_dev.jsonl")
N_CLAIMS = 4  # each claim = 1 LLM verdict call; ~5-8 LLM calls total
TOP_K = 5  # candidate evidence passages retrieved per claim
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load — claims (deterministic head) and the full SciFact corpus

The corpus is JSONL: each line is a record with `doc_id`, `title`, and `abstract` (a list of sentence strings). One searchable string per doc is built as `title. <abstract sentences>`. The claims dev set carries `id` and `claim` fields; `head(n)` keeps the first `n` rows so every run works on the same slice.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — claims (deterministic head) and the full SciFact corpus
# --------------------------------------------------------------------------
def load_corpus(path: Path) -> list[dict]:
    """All corpus records: ``{"doc_id", "title", "abstract"}``."""
    docs: list[dict] = []
    with open(path) as f:
        for line in f:
            docs.append(json.loads(line))
    return docs


def corpus_texts(docs: list[dict]) -> list[str]:
    """One searchable string per doc: ``title. <abstract sentences>``."""
    return [f"{doc['title']}. {' '.join(doc['abstract'])}" for doc in docs]


def load_claims(path: Path, n: int) -> list[dict]:
    """First ``n`` claims from the dev set (each carries its ``id``)."""
    claims: list[dict] = []
    with open(path) as f:
        for line in f:
            claims.append(json.loads(line))
            if len(claims) >= n:
                break
    return claims


## 3. Experiment — embed the corpus once, then verify each claim

This is the evidence-verification pipeline, in two halves:

- **Embed** — the 5183-doc corpus is embedded ONCE with BGE (`~5-6 min on CPU`) and the vectors are kept in the experiment dict so every claim reuses them. Per-claim retrieval is then a pure cosine rank over cached vectors, not a re-embed.
- **Verify** — `verify_loop` (in `tools/verifier.py`) takes each claim, retrieves top-5 evidence by cosine similarity, sends them to the local LLM via `json_object`, and returns a three-way verdict: `SUPPORTED`, `REFUTED`, or `NOT_ENOUGH_INFO`. The parse-failure fallback guarantees the pipeline never crashes on a stubborn local model.

The result is a single `exp` dict holding every claim's verdict, reason, evidence indices, and the verdict distribution — the artifact labs 04 and 05 build on.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — embed the corpus once, then verify each claim
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    corpus = load_corpus(CORPUS_PATH)
    texts = corpus_texts(corpus)
    claims = load_claims(CLAIMS_PATH, N_CLAIMS)

    llm = OllamaLLM()  # local qwen2.5-coder:7b; json_object returns the verdict
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device=BGE_DEVICE)

    # Embed the 5183-doc corpus ONCE (~5-6 min on CPU) and keep the vectors in
    # the experiment dict so every claim reuses them.
    t0 = time.perf_counter()
    corpus_embeddings = embedder.embed_documents(texts)
    embed_s = time.perf_counter() - t0

    results = verify_loop(
        llm,
        embedder,
        [claim["claim"] for claim in claims],
        texts,
        top_k=TOP_K,
        progress=lambda done, total: print(
            f"  verified {done}/{total} claims", flush=True
        ),
        corpus_embeddings=corpus_embeddings,
    )
    total_s = time.perf_counter() - t0

    rows = []
    for claim, result in zip(claims, results):
        rows.append(
            {
                "id": claim["id"],
                "claim": result["claim"],
                "verdict": result["verdict"],
                "reason": result["reason"],
                "evidence_indices": result["evidence_indices"],
                "evidence_titles": [
                    corpus[i]["title"] for i in result["evidence_indices"]
                ],
            }
        )
    return {
        "rows": rows,
        "corpus_size": len(corpus),
        "embed_s": embed_s,
        "total_s": total_s,
        "corpus_embeddings": corpus_embeddings,  # cached for reuse/inspection
        "distribution": dict(Counter(row["verdict"] for row in rows)),
    }


## 4. Demo

The demo prints the artifact from five angles: per-claim id/claim/evidence-titles/verdict/reason, the verdict distribution across the 4 claims, and the takeaway. The verdicts are the interesting part — a `SUPPORTED` verdict means the retrieved evidence backs the claim, `REFUTED` means it contradicts, and `NOT_ENOUGH_INFO` means the evidence is irrelevant. Parse-failure fallback never crashes the pipeline; it surfaces as a recognizable reason string. Agreement with SciFact gold labels is intentionally not enforced in the gate.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 10-03 — Evidence verification loop (SciFact claims)")
    print(f"{len(exp['rows'])} claims over {exp['corpus_size']} docs; "
          f"embed {exp['embed_s']:.0f}s, total {exp['total_s']:.0f}s")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        print(f"\nC{i} ({row['id']}): {row['claim'][:100]}")
        for title in row["evidence_titles"][:3]:
            print(f"    evidence: {title[:95]}")
        print(f"    verdict : {row['verdict']}")
        print(f"    reason  : {row['reason'][:140]}")

    print(f"\n[4] Verdict distribution over {len(exp['rows'])} claims")
    for verdict in ("SUPPORTED", "REFUTED", "NOT_ENOUGH_INFO"):
        print(f"    {verdict:<16} {exp['distribution'].get(verdict, 0)}")
    print(f"    {'TOTAL':<16} {len(exp['rows'])}")

    print(f"\n[5] Takeaway")
    print("    Verification is RAG's answer-quality gate: retrieve the most")
    print("    plausible evidence, then ask the model whether it actually")
    print("    supports, refutes, or ignores the claim. The verdict is a")
    print("    machine-checkable signal (not free text), and the 'parse")
    print("    failure' fallback guarantees the pipeline never crashes on a")
    print("    stubborn local model. Agreement with SciFact gold labels is")
    print("    intentionally not enforced in the gate.")


## 5. Verification gate

The lab ships a `--verify` gate: hard checks the experiment must clear — all 4 claims were verified (the loop terminated), every verdict is one of the three valid values, every verdict has a non-empty reason (or the explicit parse-failure note), and every claim retrieved exactly TOP_K evidence passages. The gate turns "the lab ran" into "the lab ran *correctly*" — the same discipline every lab in this repo applies.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    rows = exp["rows"]

    checks.append((f"all {N_CLAIMS} claims were verified (loop terminated)",
                   len(rows) == N_CLAIMS))
    checks.append(("every verdict is one of SUPPORTED/REFUTED/NOT_ENOUGH_INFO",
                   all(row["verdict"] in VALID_VERDICTS for row in rows)))
    checks.append(("every verdict has a non-empty reason (or a parse-failure note)",
                   all(row["reason"].strip() for row in rows)))
    checks.append((f"every claim retrieved {TOP_K} evidence passages",
                   all(len(row["evidence_indices"]) == TOP_K for row in rows)))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Expect ~6-10 minutes total: the 5183-doc CPU embed dominates at ~5-6 min, and the per-claim LLM verdicts add only ~1-4 min (~5-8 calls). No downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Per-claim verdicts, evidence titles, and the distribution — the machine-checkable signals the verification loop produced.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check that Ollama is serving `qwen2.5-coder:7b` and that both SciFact data files are intact.


In [ ]:
verify_gate(exp)
